# 11. Tumor Neighbor Analysis

**Description:** Identifies tumor cells neighboring specific CAF subtypes using Delaunay spatial graphs, then performs differential gene expression and pathway enrichment analysis to characterize the transcriptomic differences of CAF-neighboring tumor cells.

**Analyses included:**
- NR CAF neighbor analysis (Delaunay spatial graph, DEGs, pathway enrichment)
- Per-patient pathway analysis (MSigDB Hallmark, non-canonical WNT)
- CR CAF neighbor analysis (in NR and MPR patients)
- Other CAF neighbor analysis
- Per-patient CAF gridding and DEG analysis

**Conda environment:** `skny`

## Imports

In [ ]:
import os
import re
import math
import warnings

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import squidpy as sq
import stlearn as st
import gseapy as gp
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import seaborn as sns
from matplotlib.lines import Line2D
from adjustText import adjust_text
from gseapy import Msigdb, barplot, dotplot
from statsmodels.stats.multitest import multipletests
from concurrent.futures import ProcessPoolExecutor, as_completed
from PIL import Image
import cv2
import networkx as nx

warnings.filterwarnings("ignore")

## Configuration

In [ ]:
DATA_DIR = "../data"
RESULTS_DIR = "../results"
FIGURES_DIR = "../figures"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# DEG thresholds
PVAL_THRESHOLD = 1e-2
LOGFC_THRESHOLD = 0.5

# CAF subtype definitions
NR_CAF_SUBCLUSTERS = ["EEF1G+ CAF", "MMP14+ CAF", "MMP11+ CAF"]
CR_CAF_SUBCLUSTERS = ["CCDC80+ CAF"]
OTHER_CAF_SUBCLUSTERS = [
    "GREM1+ CAF", "ADAMDEC1+ CAF", "ANXA2+ CAF", "THBS1+ CAF",
    "L1CAM+ CAF", "CCDC80+ CAF", "GATA2+ CAF", "CCN1+ CAF",
    "WNT5A+ CAF", "CSRP1+ CAF", "SOST+ CAF", "SFRP4+ CAF", "STAT1+ CAF",
]

PATIENTS = ["Pt-1", "Pt-2", "Pt-3", "Pt-4", "Pt-5", "Pt-6", "Pt-7", "Pt-8"]

## Utility Functions

### Generalized CAF neighbor identification

In [ ]:
def identify_caf_neighbors(adata, caf_subclusters, caf_label, tumor_subcluster="Tumor"):
    """Identify tumor cells neighboring a specific CAF subtype using Delaunay triangulation.
    
    Parameters
    ----------
    adata : AnnData
        Spatial transcriptomics data for a single sample.
    caf_subclusters : list of str
        Subcluster names that define this CAF subtype.
    caf_label : str
        Label for this CAF subtype (e.g. "NR CAF", "CR CAF").
    tumor_subcluster : str
        Name of the tumor subcluster column value.
    
    Returns
    -------
    adata : AnnData
        Updated with neighbor annotations.
    neighbor_col : str
        Name of the neighbor annotation column added.
    """
    adata.obs["CAF_annotation"] = "Other"
    adata.obs.loc[
        adata.obs["subcluster"].isin(caf_subclusters), "CAF_annotation"
    ] = caf_label

    sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True)

    target_cells = adata.obs_names[adata.obs["CAF_annotation"] == caf_label]
    adj = adata.obsp["spatial_connectivities"]

    is_neighbor = np.zeros(adata.n_obs, dtype=bool)
    for cell in target_cells:
        idx = adata.obs_names.get_loc(cell)
        neighbors = adj[idx].nonzero()[1]
        is_neighbor[neighbors] = True

    # Exclude the CAF cells themselves
    is_neighbor[adata.obs["CAF_annotation"] == caf_label] = False
    adata.obs[f"neighbor_of_{caf_label}"] = is_neighbor

    # Annotate tumor cells as Neighbor / Not_neighbor / Other
    neighbor_col = f"{caf_label}_neighbor_Tumor"
    adata.obs.loc[
        adata.obs[f"neighbor_of_{caf_label}"] & (adata.obs["subcluster"] == tumor_subcluster),
        neighbor_col,
    ] = "Neighbor"
    adata.obs.loc[
        ~adata.obs[f"neighbor_of_{caf_label}"] & (adata.obs["subcluster"] == tumor_subcluster),
        neighbor_col,
    ] = "Not_neighbor"
    adata.obs[neighbor_col] = adata.obs[neighbor_col].fillna("Other")

    return adata, neighbor_col

### DEG classification

In [ ]:
def classify_degs(degs, pval_threshold=PVAL_THRESHOLD, logfc_threshold=LOGFC_THRESHOLD,
                  up_color="red", down_color="black"):
    """Classify DEGs into up/down/neutral based on thresholds."""
    degs = degs.copy()
    degs["color"] = "grey"
    degs.loc[
        (degs["pvals_adj"] < pval_threshold) & (degs["logfoldchanges"] > logfc_threshold),
        "color",
    ] = up_color
    degs.loc[
        (degs["pvals_adj"] < pval_threshold) & (degs["logfoldchanges"] < -logfc_threshold),
        "color",
    ] = down_color
    degs["pvals_adj"] = degs["pvals_adj"].replace(0, 1e-300)
    return degs

### Volcano plot

In [ ]:
def plot_volcano(degs, group_name, reference_name, save_path=None,
                 xlim=(-3, 3), n_top_up=4, n_top_down=20,
                 additional_genes=None, up_color="red", down_color="black"):
    """Generate a volcano plot from classified DEGs."""
    plt.figure(figsize=(7, 5))

    for color, group in degs.groupby("color"):
        plt.scatter(
            group["logfoldchanges"],
            -np.log10(group["pvals_adj"]),
            color=color, alpha=0.7, label=color,
        )

    plt.xlim(*xlim)

    top_genes_up = degs[degs["logfoldchanges"] > LOGFC_THRESHOLD].sort_values("pvals_adj").head(n_top_up)
    top_genes_down = degs[degs["logfoldchanges"] < -LOGFC_THRESHOLD].sort_values("pvals_adj").head(n_top_down)

    texts = []
    for _, row in top_genes_up.iterrows():
        if row["color"] == up_color:
            texts.append(plt.text(row["logfoldchanges"], -np.log10(row["pvals_adj"]), row["names"], fontsize=10))
    for _, row in top_genes_down.iterrows():
        if row["color"] == down_color:
            texts.append(plt.text(row["logfoldchanges"], -np.log10(row["pvals_adj"]), row["names"], fontsize=10))

    if additional_genes:
        for _, row in degs[degs["names"].isin(additional_genes)].iterrows():
            texts.append(plt.text(row["logfoldchanges"], -np.log10(row["pvals_adj"]), row["names"], fontsize=10))

    if texts:
        adjust_text(
            texts,
            force_text=100, force_points=100, expand_text=(1.2, 1.5),
            arrowprops=dict(arrowstyle="->", color="gray", lw=1.0),
        )

    plt.xlabel("Log2 Fold Change", fontsize=16)
    plt.ylabel("-Log10 Adjusted p-value", fontsize=16)
    plt.legend(
        [reference_name, "Neutral", group_name],
        loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=12,
    )
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()

### Pathway enrichment bubble plot

In [ ]:
def parse_overlap(ov):
    """Parse Enrichr overlap string to percentage."""
    try:
        hit, total = map(int, ov.split("/"))
        return 100 * hit / total
    except Exception:
        return np.nan


def plot_pathway_bubble(df, gene_set_filter=None, term_filter=None,
                        figsize=(5, 8), max_pval_color=10,
                        group_col="group", save_path=None):
    """Generate a pathway enrichment bubble plot.
    
    Parameters
    ----------
    df : DataFrame
        Enrichment results with columns: Term, Overlap, P-value, Gene_set, group.
    gene_set_filter : str, optional
        Filter to a specific gene set library.
    term_filter : list, optional
        Filter to specific terms.
    """
    df = df.copy()
    df["Overlap_pct"] = df["Overlap"].apply(parse_overlap)
    df["adj_pval"] = multipletests(df["P-value"], method="fdr_bh")[1]
    df["-log10(adj_pval)"] = -np.log10(df["adj_pval"])
    df = df.sort_values([group_col])

    top_terms = df[df["adj_pval"] < 0.05]["Term"].unique()
    df_top = df[df["Term"].isin(top_terms)]

    if gene_set_filter:
        df_top = df_top[df_top["Gene_set"] == gene_set_filter]
    if term_filter:
        df_top = df_top[df_top["Term"].isin(term_filter)]

    if df_top.empty:
        print("No significant terms to plot.")
        return df_top

    group_order = sorted(df_top[group_col].unique())
    df_top[group_col] = pd.Categorical(df_top[group_col], categories=group_order, ordered=True)

    term_order = df_top.groupby("Term")["Overlap_pct"].mean().sort_values().index
    df_top["Term"] = pd.Categorical(df_top["Term"], categories=term_order, ordered=True)

    norm = plt.Normalize(df_top["-log10(adj_pval)"].min(), max_pval_color)
    cmap = plt.cm.Reds

    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(1, 3, width_ratios=[6, 0.8, 0.3], wspace=0.6)

    ax = fig.add_subplot(gs[0])
    ax.set_xticks(range(len(group_order)))
    ax.set_xticklabels(group_order, rotation=90, ha="center", fontsize=12)
    ax.set_yticks(range(len(term_order)))
    ax.set_yticklabels(term_order, fontsize=12)

    group_map = {g: i for i, g in enumerate(group_order)}
    term_map = {t: i for i, t in enumerate(term_order)}

    for _, row in df_top.iterrows():
        color = cmap(norm(row["-log10(adj_pval)"]))
        linewidth = 2 if row["adj_pval"] < 0.05 else 0.5
        ax.scatter(
            group_map[row[group_col]], term_map[row["Term"]],
            s=row["Overlap_pct"] * 7, c=[color],
            edgecolors="black", linewidths=linewidth, alpha=1,
        )

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.grid()
    ax.set_axisbelow(True)

    y_min, y_max = ax.get_ylim()
    ax.set_ylim(y_min + 0.7, y_max - 0.5)

    # Size legend
    legend_ax = fig.add_subplot(gs[1])
    legend_ax.axis("off")
    sizes = [0, 1, 5, 10, 20]
    markers = [
        Line2D([0], [0], marker="o", color="w",
               markerfacecolor="gray", markersize=np.sqrt(s * 7),
               markeredgecolor="black", label=f"{s}%")
        for s in sizes
    ]
    legend_ax.legend(handles=markers, title="Overlap (%)", loc="center", frameon=False, fontsize=12)

    # Colorbar
    sm = cm.ScalarMappable(norm=norm, cmap="Reds")
    sm.set_array([])
    cbar_ax = fig.add_axes([0.71, 0.65, 0.04, 0.2])
    cbar = plt.colorbar(sm, cax=cbar_ax)
    cbar.set_label("-log10(Adj. P-value)", loc="center", fontsize=12)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    return df_top

### calculate_distance (for gridding analysis)

In [ ]:
def cv2pil(image):
    """Convert OpenCV image to PIL Image."""
    new_image = image.copy()
    if new_image.ndim == 2:
        pass
    elif new_image.shape[2] == 3:
        new_image = cv2.cvtColor(new_image, cv2.COLOR_BGR2RGB)
    elif new_image.shape[2] == 4:
        new_image = cv2.cvtColor(new_image, cv2.COLOR_BGRA2RGBA)
    return Image.fromarray(new_image)


def calculate_distance(grid, pos_marker_ls=None, neg_marker_ls=None, use_label=None):
    """Calculate shortest distance from tumor surface for each grid cell.
    
    Parameters
    ----------
    grid : AnnData
        Gridded spatial data.
    pos_marker_ls, neg_marker_ls : list, optional
        Positive/negative marker lists for defining tumor region.
    use_label : str, optional
        Column in grid.obs to use as label (alternative to marker lists).
    
    Returns
    -------
    grid : AnnData
        Updated with distance information in grid.shortest.
    """
    N_ROW = len(grid.uns["grid_yedges"]) - 1
    N_COL = len(grid.uns["grid_xedges"]) - 1

    if use_label is None:
        pos_series = sum([grid.to_df()[i] for i in pos_marker_ls])
        pos_series = pd.Series([1 if i >= 1 else 0 for i in pos_series], index=pos_series.index)
        if neg_marker_ls is not None:
            neg_series = sum([grid.to_df()[i] for i in neg_marker_ls])
            neg_series = pd.Series([1 if i >= 1 else 0 for i in neg_series], index=neg_series.index)
            pos_series = pos_series - neg_series
            pos_series = pd.Series([-1 if i == 0 else i for i in pos_series], index=pos_series.index)
        else:
            pos_series = pd.Series([-1 if i == 0 else i for i in pos_series], index=pos_series.index)
        pos_series.name = "tumor_grid"
        df_grid_tumor = pd.merge(
            pd.DataFrame(index=["grid_" + str(i + 1) for i in range(N_ROW * N_COL)]),
            pos_series, right_index=True, left_index=True, how="left",
        ).fillna(-1)
    else:
        pos_series = grid.obs[use_label].copy()
        pos_series.name = "tumor_grid"
        df_grid_tumor = pd.merge(
            pd.DataFrame(index=["grid_" + str(i + 1) for i in range(N_ROW * N_COL)]),
            pos_series, right_index=True, left_index=True, how="left",
        ).fillna(-1)

    # Generate image of positive grid
    fig, ax = plt.subplots(figsize=(N_COL, N_ROW), dpi=1, tight_layout=True)
    cmap_img = matplotlib.cm.viridis
    cmap_img.set_bad("black", 1.0)
    cmap_img.set_under(color="black")
    ax.imshow(
        np.array(df_grid_tumor["tumor_grid"]).reshape(N_COL, N_ROW).T,
        cmap=cmap_img, vmin=0, vmax=1,
    )
    ax.axis("off")
    fig.canvas.draw()
    img = np.array(fig.canvas.renderer.buffer_rgba())
    img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
    plt.close()
    grid.uns["marker"] = img.copy()

    # Median + Gaussian filter
    img_med = cv2.medianBlur(img, ksize=3)
    img_med = cv2.GaussianBlur(img_med, (3, 3), 0)
    grid.uns["marker_median"] = img_med.copy()

    # Binary threshold and contour detection
    threshold = 20
    img_gray = cv2.cvtColor(img_med, cv2.COLOR_BGR2GRAY)
    ret, img_binary = cv2.threshold(img_gray, threshold, 255, cv2.THRESH_BINARY)
    grid.uns["marker_binary"] = img_binary.copy()
    contours, hierarchy = cv2.findContours(img_binary, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    contours_ = list(contours)

    img_binary = cv2.cvtColor(img_binary, cv2.COLOR_GRAY2BGR)
    img_binary_ = np.where(img_binary == (255, 255, 255), (36, 231, 253), (0, 0, 0))
    img_binary_ = np.array(img_binary_, dtype=np.uint8)
    img_color_with_contours = cv2.drawContours(img_binary_, contours_, -1, (0, 255, 0), 1)
    grid.uns["marker_delineation"] = img_color_with_contours

    # Convert image to graph structure
    image = cv2pil(img_color_with_contours)
    width, height = image.size
    graph = nx.Graph()

    tumor_contour_pixel_ls = []
    tumor_pixel_ls = []
    edge_pixel_ls = []

    for y in range(height):
        for x in range(width):
            pixel_value = image.getpixel((x, y))
            graph.add_node((x, y), color=pixel_value)
            if pixel_value == (0, 255, 0):
                tumor_contour_pixel_ls.append((x, y))
            elif pixel_value == (253, 231, 36):
                tumor_pixel_ls.append((x, y))
            if y == 0 or y == height - 1 or x == 0 or x == width - 1:
                edge_pixel_ls.append((x, y))

    nodes_ls = list(graph.nodes)

    inside_contour_ls = []
    for node in nodes_ls:
        for i in contours_:
            if cv2.pointPolygonTest(i, node, False) == 1:
                inside_contour_ls.append(node)
                break

    tumor_pixel_ls = [i for i in tumor_pixel_ls if i in inside_contour_ls]

    # Create edges between adjacent pixels
    for y in range(height):
        for x in range(width):
            current_node = (x, y)
            for neighbor in [(x, y - 1), (x, y + 1), (x - 1, y), (x + 1, y)]:
                if neighbor in graph.nodes:
                    graph.add_edge(current_node, neighbor)
            if y != height and x != width:
                for n1, n2 in [((x, y), (x + 1, y + 1)), ((x + 1, y), (x, y + 1))]:
                    distance = math.sqrt((n1[0] - n2[0]) ** 2 + (n1[1] - n2[1]) ** 2)
                    graph.add_edge(n1, n2, weight=distance)

    tumor_pixel_ls = list(
        set(tumor_pixel_ls) | (set(tumor_contour_pixel_ls) & set(edge_pixel_ls))
    )
    tumor_contour_pixel_ls = list(
        set(tumor_contour_pixel_ls) - (set(tumor_contour_pixel_ls) & set(edge_pixel_ls))
    )

    # Calculate shortest paths from tumor surface
    shortest_paths = nx.multi_source_dijkstra(graph, tumor_contour_pixel_ls, weight="weight")
    df_shotest = pd.DataFrame.from_dict(shortest_paths[0], orient="index", columns=["euclidean"])
    df_shotest.loc[df_shotest.index[df_shotest.index.isin(tumor_pixel_ls)]] *= -1
    df_shotest["euclidean_round"] = df_shotest["euclidean"].round(0)

    df_nodes = pd.DataFrame(index=nodes_ls)
    df_shotest = pd.merge(df_nodes, df_shotest, right_index=True, left_index=True, how="left").fillna(np.nan)

    fig.canvas.draw()
    img = np.array(fig.canvas.renderer.buffer_rgba())
    plt.close()
    grid.uns["shotest"] = img

    # Delineation visualizations
    col_ls = []
    for i in df_shotest["euclidean_round"]:
        if i == 0:
            col_ls.append((0, 255, 0))
        elif i < 0:
            col_ls.append((0, 255, 255))
        else:
            col_ls.append((0, 0, 0))
    grid.uns["marker_median_delineation"] = np.array(col_ls, dtype=np.uint8).reshape(N_ROW, N_COL, 3)

    col_ls = []
    for i in df_shotest["euclidean_round"]:
        if i == 0:
            col_ls.append((0, 255, 0))
        elif (i % 3 == 0) and (i > 0):
            col_ls.append((0, 0, 255))
        elif (i % 3 == 0) and (i < 0):
            col_ls.append((0, 150, 255))
        elif i < 0:
            col_ls.append((0, 255, 255))
        else:
            col_ls.append((0, 0, 0))
    grid.uns["shotest_30_delineation"] = np.array(col_ls, dtype=np.uint8).reshape(N_ROW, N_COL, 3)

    # Discretization into distance bins
    df_shotest["region"] = pd.cut(
        df_shotest.dropna()["euclidean"],
        bins=list(range(-15, 31, 3)),
    )
    for i in df_shotest.dropna().region.value_counts()[df_shotest.dropna().region.value_counts() > 100].sort_index().index:
        col_ls = [(255, 255, 255) if s == i else (0, 0, 0) for s in df_shotest["region"]]
        grid.uns[f"shotest_region_{i}_delineation"] = np.array(col_ls, dtype=np.uint8).reshape(N_ROW, N_COL, 3)

    setattr(grid, "shortest", df_shotest)
    return grid

## Data Loading

In [ ]:
adata_full = ad.read_h5ad(os.path.join(DATA_DIR, "bicrc_integrated_annotated.h5ad"))
print(f"Full dataset: {adata_full.shape}")

## NR CAF Neighbor Analysis

### Subset to NR (SD) resection samples

In [ ]:
adata_nr = adata_full[
    (adata_full.obs["Timepoint"] == "Resection") & (adata_full.obs["Response"] == "SD")
].copy()
print(f"NR resection subset: {adata_nr.shape}")

### Neighbor identification (Delaunay spatial graph)

In [ ]:
df_neighbor_NR_CAF = pd.DataFrame()
for sample in adata_nr.obs["Sample"].unique():
    adata_sample = adata_nr[adata_nr.obs["Sample"] == sample].copy()
    adata_sample, neighbor_col = identify_caf_neighbors(
        adata_sample, NR_CAF_SUBCLUSTERS, "NR CAF"
    )

    sq.pl.spatial_scatter(
        adata_sample,
        library_id="spatial", shape=None,
        color=[neighbor_col],
        wspace=0.4,
        palette=matplotlib.colors.ListedColormap(["yellow", "red", "blue"]),
        save=os.path.join(FIGURES_DIR, f"neighbor_NR_CAF_{sample}.png"),
    )
    df_neighbor_NR_CAF = pd.concat([
        df_neighbor_NR_CAF, adata_sample.obs[[neighbor_col]]
    ])

### Merge neighbor annotations back

In [ ]:
adata_nr.obs = pd.merge(
    adata_nr.obs, df_neighbor_NR_CAF,
    right_index=True, left_index=True, how="left",
)

adata_nr.obs["CAF_annotation"] = "Other"
adata_nr.obs.loc[
    adata_nr.obs["subcluster"].isin(NR_CAF_SUBCLUSTERS), "CAF_annotation"
] = "NR CAF"

adata_nr.obs["NR_CAF_neighbor_Tumor_CAF"] = adata_nr.obs["NR CAF_neighbor_Tumor"]
adata_nr.obs["NR_CAF_neighbor_Tumor_CAF"] = adata_nr.obs["NR_CAF_neighbor_Tumor_CAF"].astype(str)
adata_nr.obs.loc[adata_nr.obs["CAF_annotation"] == "NR CAF", "NR_CAF_neighbor_Tumor_CAF"] = "NR CAF"
adata_nr.obs["NR_CAF_neighbor_Tumor_CAF"] = adata_nr.obs["NR_CAF_neighbor_Tumor_CAF"].replace(
    "Not_neighbor", "Non_neighbor"
)
adata_nr = adata_nr[adata_nr.obs["NR_CAF_neighbor_Tumor_CAF"].sort_values(ascending=False).index]

### Spatial visualization (example patient)

In [ ]:
sq.pl.spatial_scatter(
    adata_nr[
        (adata_nr.obs["Sample"] == "Pt-7_Resection")
        & (adata_nr.obs["NR_CAF_neighbor_Tumor_CAF"] != "Other")
    ],
    library_id="spatial", shape=None, size=1,
    color=["NR_CAF_neighbor_Tumor_CAF"],
    wspace=0.4,
    palette=matplotlib.colors.ListedColormap(["#DA9264", "red", "black"]),
)

### Per-cell pathway enrichment (MSigDB Hallmark)

In [ ]:
adata_sample_enr = adata_nr[
    (adata_nr.obs["Sample"] == "Pt-1_Resection")
    & (
        (adata_nr.obs["NR_CAF_neighbor_Tumor_CAF"] == "Neighbor")
        | (adata_nr.obs["NR_CAF_neighbor_Tumor_CAF"] == "Non_neighbor")
    )
]
df_exp = adata_sample_enr.to_df()

In [ ]:
# Download MSigDB Hallmark gene set (run once)
import requests

gmt_path = os.path.join(DATA_DIR, "MSigDB_Hallmark_2020.gmt")
if not os.path.exists(gmt_path):
    url = "https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=MSigDB_Hallmark_2020"
    response = requests.get(url)
    with open(gmt_path, "w") as f:
        f.write(response.text)
    print("Downloaded MSigDB_Hallmark_2020.gmt")
else:
    print("MSigDB_Hallmark_2020.gmt already exists")

In [ ]:
# Per-cell enrichment using top 5% expressed genes
gene_thresholds = df_exp.quantile(0.95, axis=0)

def run_enrichr(i_row):
    i, exp_cell = i_row
    high_exp_genes = exp_cell[exp_cell > gene_thresholds].index.tolist()
    if not high_exp_genes:
        return None
    enr = gp.enrichr(
        gene_list=high_exp_genes,
        gene_sets=gmt_path,
        organism="human",
        outdir=None,
    )
    df_temp = enr.results.copy()
    df_temp["cell"] = i
    return df_temp

df_res_list = []
with ProcessPoolExecutor() as executor:
    futures = [executor.submit(run_enrichr, (i, df_exp.loc[i, :])) for i in df_exp.index]
    for future in as_completed(futures):
        result = future.result()
        if result is not None:
            df_res_list.append(result)

df_res = pd.concat(df_res_list, ignore_index=True)
df_res = df_res.set_index("cell")
print(f"Enrichment results: {df_res.shape[0]} entries across {df_res.index.nunique()} cells")

### Pathway spatial visualization

In [ ]:
df_res["Log_adjusted P-value"] = np.log10(df_res["Adjusted P-value"]) * -1

term = "Epithelial Mesenchymal Transition"
score = "Log_adjusted P-value"

df_res_pathway = df_res[df_res["Term"] == term][[score]]
df_res_pathway = df_res_pathway.rename(columns={score: term})

adata_sample_enr.obs = pd.merge(
    adata_sample_enr.obs, df_res_pathway,
    right_index=True, left_index=True, how="left",
)

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "gray"

adata_sorted = adata_sample_enr[adata_sample_enr.obs[term].sort_values(ascending=True).index].copy()

sq.pl.spatial_scatter(
    adata_sorted,
    library_id="spatial", shape=None, size=1,
    color=[term], cmap="magma",
    wspace=0.4, vmin=0, vmax=2.0,
)

### DEG analysis (Neighbor vs Non_neighbor)

In [ ]:
group_name = "Neighbor"
sc.tl.rank_genes_groups(
    adata_nr, "NR_CAF_neighbor_Tumor_CAF",
    groups=[group_name], reference="Non_neighbor", method="wilcoxon",
)
degs_nr = sc.get.rank_genes_groups_df(adata_nr, group=group_name)
degs_nr = classify_degs(degs_nr)

print(f"Up-regulated: {(degs_nr['color'] == 'red').sum()}")
print(f"Down-regulated: {(degs_nr['color'] == 'black').sum()}")

### Volcano plot

In [ ]:
additional_genes_nr = [
    "HOXD9", "HOXD10",
    "TNFRSF6B", "TNFRSF11B", "CDKN2A", "NFKB2", "TNFRSF9",
    "SMAD3", "PDGFA", "PDGFB", "ITGA3", "ITGB4", "LAMC2", "SDC4", "ITGB6",
]

plot_volcano(
    degs_nr, group_name="Neighbor", reference_name="Non_neighbor",
    additional_genes=additional_genes_nr,
    save_path=os.path.join(FIGURES_DIR, "Neighbor_NR_CAF_neighbor_volcano_integrate.png"),
)

## Per-Patient Pathway Analysis

### Per-patient DEGs and pathway enrichment

In [ ]:
df_patient_pathway = pd.DataFrame()
for sample in adata_nr.obs["Sample"].unique():
    adata_sample = adata_nr[adata_nr.obs["Sample"] == sample]

    sc.tl.rank_genes_groups(
        adata_sample, "NR_CAF_neighbor_Tumor_CAF",
        groups=["Neighbor"], reference="Non_neighbor", method="wilcoxon",
    )
    degs = sc.get.rank_genes_groups_df(adata_sample, group="Neighbor")
    degs = classify_degs(degs)
    deg_ls = degs[degs["color"] == "red"]["names"].tolist()

    if len(deg_ls) == 0:
        continue

    enr = gp.enrichr(
        gene_list=deg_ls,
        gene_sets=["MSigDB_Hallmark_2020", "KEGG_2021_Human"],
        organism="human",
        outdir=None,
    )
    df_temp = enr.results.copy()
    df_temp["group"] = sample
    df_patient_pathway = pd.concat([df_patient_pathway, df_temp])

df_patient_pathway["group"] = [i.split("_")[0] for i in df_patient_pathway["group"]]
print(f"Patient pathway results: {df_patient_pathway.shape}")

### Pathway enrichment heatmap (selected pathways)

In [ ]:
SELECTED_PATHWAYS = [
    "KRAS Signaling Up",
    "IL-6/JAK/STAT3 Signaling",
    "TNF-alpha Signaling via NF-kB",
    "Hedgehog Signaling",
    "Angiogenesis",
    "Interferon Gamma Response",
    "IL-2/STAT5 Signaling",
    "Cholesterol Homeostasis",
    "Hypoxia",
    "mTORC1 Signaling",
    "Notch Signaling",
    "Interferon Alpha Response",
    "p53 Pathway",
    "Epithelial Mesenchymal Transition",
    "Wnt-beta Catenin Signaling",
    "TGF-beta Signaling",
]

df_top_selected = plot_pathway_bubble(
    df_patient_pathway,
    gene_set_filter="MSigDB_Hallmark_2020",
    term_filter=SELECTED_PATHWAYS,
    figsize=(5, 6),
    save_path=os.path.join(FIGURES_DIR, "per_patient_pathway_heatmap_selected.png"),
)

### Non-canonical WNT pathway analysis

In [ ]:
ENRICHR_LIBS_WNT = [
    "Reactome_2022",
    "WikiPathways_2024_Human",
    "GO_Biological_Process_2025",
]

NONCAN_WNT_PAT = re.compile(
    r"(non[-\s]?canonical\s+wnt|wnt.?pcp|planar\s+cell\s+polarity|wnt.?calcium|"
    r"ror2|ryk|vangl|prickle|daam1|rho|rac|jnk|wnt5a|wnt11)",
    flags=re.IGNORECASE,
)

df_ncwnt = pd.DataFrame()
for sample in adata_nr.obs["Sample"].unique():
    adata_sample = adata_nr[adata_nr.obs["Sample"] == sample]

    sc.tl.rank_genes_groups(
        adata_sample, "NR_CAF_neighbor_Tumor_CAF",
        groups=["Neighbor"], reference="Non_neighbor", method="wilcoxon",
    )
    degs = sc.get.rank_genes_groups_df(adata_sample, group="Neighbor")
    degs["pvals_adj"] = degs["pvals_adj"].replace(0, 1e-300)

    deg_up = degs[
        (degs["pvals_adj"] < PVAL_THRESHOLD) & (degs["logfoldchanges"] > LOGFC_THRESHOLD)
    ]
    deg_ls = deg_up["names"].tolist()

    if len(deg_ls) < 5:
        continue

    enr = gp.enrichr(
        gene_list=deg_ls,
        gene_sets=ENRICHR_LIBS_WNT,
        organism="human",
        outdir=None,
    )
    res = enr.results.copy()
    res["Sample"] = sample
    res["is_noncanonical_wnt"] = res["Term"].astype(str).apply(lambda s: bool(NONCAN_WNT_PAT.search(s)))
    df_ncwnt = pd.concat([df_ncwnt, res[res["is_noncanonical_wnt"]]], ignore_index=True)

print(f"Non-canonical WNT results: {df_ncwnt.shape}")
df_ncwnt.head(10)

In [ ]:
# Broader non-canonical WNT search with more libraries
libs = gp.get_library_name(organism="Human")
pat_lib = re.compile(
    r"(Reactome|WikiPathways|NCI|PID|BioCarta|KEGG|GO_Biological_Process|Panther|Pathway)", re.I
)
cand_libs = [x for x in libs if pat_lib.search(x)]

TERM_PAT = re.compile(
    r"(non[-\s]?canonical\s+wnt|beta[-\s]?catenin\s+independent|"
    r"wnt.?pcp|planar\s+cell\s+polarity|wnt.?calcium|"
    r"ror2|ryk|vangl|prickle|daam1|rho|rac|jnk|wnt5a|wnt11)",
    flags=re.IGNORECASE,
)

df_nc_broad = []
for sample in adata_nr.obs["Sample"].unique():
    adata_sample = adata_nr[adata_nr.obs["Sample"] == sample]

    sc.tl.rank_genes_groups(
        adata_sample, "NR_CAF_neighbor_Tumor_CAF",
        groups=["Neighbor"], reference="Non_neighbor", method="wilcoxon",
    )
    degs = sc.get.rank_genes_groups_df(adata_sample, group="Neighbor").copy()
    degs["pvals_adj"] = degs["pvals_adj"].replace(0, 1e-300)

    pval_threshold_relaxed = 5e-2
    logfc_threshold_relaxed = 0.25
    deg_up = degs[
        (degs["pvals_adj"] < pval_threshold_relaxed)
        & (degs["logfoldchanges"] > logfc_threshold_relaxed)
    ]
    gene_list = deg_up["names"].tolist()

    if len(gene_list) < 10:
        continue

    enr = gp.enrichr(
        gene_list=gene_list,
        gene_sets=cand_libs,
        organism="human",
        outdir=None,
    )
    res = enr.results.copy()
    res["Sample"] = sample
    res["is_ncWNT"] = res["Term"].astype(str).apply(lambda s: bool(TERM_PAT.search(s)))
    df_nc_broad.append(res[res["is_ncWNT"]])

df_nc_broad = pd.concat(df_nc_broad, ignore_index=True) if df_nc_broad else pd.DataFrame()
df_nc_broad["group"] = [i.split("_")[0] for i in df_nc_broad["Sample"]]
print(f"Broad non-canonical WNT results: {df_nc_broad.shape}")

In [ ]:
# WNT-specific pathway bubble plot
if not df_nc_broad.empty:
    df_nc_wnt_filtered = df_nc_broad[df_nc_broad["Term"].str.contains("WNT|Wnt")]
    df_nc_wnt_filtered = df_nc_wnt_filtered[df_nc_wnt_filtered["Gene_set"].isin([
        "Reactome_Pathways_2024", "GO_Biological_Process_2025",
        "WikiPathways_2024_Human", "KEGG_2026", "NCI-Nature_2016", "BioCarta_2016",
    ])]
    if not df_nc_wnt_filtered.empty:
        plot_pathway_bubble(
            df_nc_wnt_filtered, figsize=(5, 5),
            save_path=os.path.join(FIGURES_DIR, "ncWNT_pathway_bubble.png"),
        )

In [ ]:
# Save per-patient pathway results
df_patient_pathway.to_csv(
    os.path.join(RESULTS_DIR, "individual_pathway_neighbor.tsv"), sep="\t"
)

## CR CAF Neighbor Analysis (in NR patients)

In [ ]:
adata_cr_nr = adata_full[
    (adata_full.obs["Timepoint"] == "Resection") & (adata_full.obs["Response"] == "SD")
].copy()

df_neighbor_CR_CAF = pd.DataFrame()
for sample in adata_cr_nr.obs["Sample"].unique():
    adata_sample = adata_cr_nr[adata_cr_nr.obs["Sample"] == sample].copy()
    adata_sample, neighbor_col = identify_caf_neighbors(
        adata_sample, CR_CAF_SUBCLUSTERS, "CR CAF"
    )

    sq.pl.spatial_scatter(
        adata_sample,
        library_id="spatial", shape=None,
        color=[neighbor_col],
        wspace=0.4,
        palette=matplotlib.colors.ListedColormap(["yellow", "red", "blue"]),
        save=os.path.join(FIGURES_DIR, f"neighbor_CR_CAF_{sample}.png"),
    )
    df_neighbor_CR_CAF = pd.concat([
        df_neighbor_CR_CAF, adata_sample.obs[[neighbor_col]]
    ])

In [ ]:
adata_cr_nr.obs = pd.merge(
    adata_cr_nr.obs, df_neighbor_CR_CAF,
    right_index=True, left_index=True, how="left",
)

adata_cr_nr.obs["CAF_annotation"] = "Other"
adata_cr_nr.obs.loc[
    adata_cr_nr.obs["subcluster"].isin(CR_CAF_SUBCLUSTERS), "CAF_annotation"
] = "CR CAF"

adata_cr_nr.obs["CR_CAF_neighbor_Tumor_CAF"] = adata_cr_nr.obs["CR CAF_neighbor_Tumor"]
adata_cr_nr.obs["CR_CAF_neighbor_Tumor_CAF"] = adata_cr_nr.obs["CR_CAF_neighbor_Tumor_CAF"].astype(str)
adata_cr_nr.obs.loc[
    adata_cr_nr.obs["CAF_annotation"] == "CR CAF", "CR_CAF_neighbor_Tumor_CAF"
] = "CR CAF"
adata_cr_nr = adata_cr_nr[adata_cr_nr.obs["CR_CAF_neighbor_Tumor_CAF"].sort_values(ascending=False).index]

adata_cr_nr.obs["CR_CAF_neighbor_Tumor_CAF"] = pd.Categorical(
    adata_cr_nr.obs["CR_CAF_neighbor_Tumor_CAF"],
    categories=["Neighbor", "CR CAF", "Not_neighbor", "Other"],
)
print(adata_cr_nr.obs["CR_CAF_neighbor_Tumor_CAF"].value_counts())

### Spatial visualization

In [ ]:
sq.pl.spatial_scatter(
    adata_cr_nr[
        (adata_cr_nr.obs["Sample"] == "Pt-7_Resection")
        & (adata_cr_nr.obs["CR_CAF_neighbor_Tumor_CAF"] != "Other")
    ],
    library_id="spatial", shape=None, size=1,
    color=["CR_CAF_neighbor_Tumor_CAF"],
    wspace=0.4,
    palette=matplotlib.colors.ListedColormap(["red", "blue", "black"]),
)

### DEG analysis + volcano plot

In [ ]:
sc.tl.rank_genes_groups(
    adata_cr_nr, "CR_CAF_neighbor_Tumor_CAF",
    groups=["Neighbor"], reference="Not_neighbor", method="wilcoxon",
)
degs_cr_nr = sc.get.rank_genes_groups_df(adata_cr_nr, group="Neighbor")
degs_cr_nr = classify_degs(degs_cr_nr)

plot_volcano(
    degs_cr_nr, group_name="Neighbor", reference_name="Not_neighbor",
    xlim=(-4, 4), n_top_up=20, n_top_down=20,
    save_path=os.path.join(FIGURES_DIR, "Neighbor_CR_CAF_neighbor_volcano_integrate.png"),
)

In [ ]:
print("CR CAF (NR patients) - upregulated DEGs:")
for gene in degs_cr_nr[degs_cr_nr["color"] == "red"]["names"]:
    print(gene)

## CR CAF Neighbor Analysis (in MPR patients)

In [ ]:
adata_cr_mpr = adata_full[
    (adata_full.obs["Timepoint"] == "Resection") & (adata_full.obs["Response"] == "MPR")
].copy()

df_neighbor_CR_CAF_mpr = pd.DataFrame()
for sample in adata_cr_mpr.obs["Sample"].unique():
    adata_sample = adata_cr_mpr[adata_cr_mpr.obs["Sample"] == sample].copy()
    adata_sample, neighbor_col = identify_caf_neighbors(
        adata_sample, CR_CAF_SUBCLUSTERS, "CR CAF"
    )

    sq.pl.spatial_scatter(
        adata_sample,
        library_id="spatial", shape=None,
        color=[neighbor_col],
        wspace=0.4,
        palette=matplotlib.colors.ListedColormap(["yellow", "red", "blue"]),
        save=os.path.join(FIGURES_DIR, f"neighbor_CR_CAF_{sample}_MPR.png"),
    )
    df_neighbor_CR_CAF_mpr = pd.concat([
        df_neighbor_CR_CAF_mpr, adata_sample.obs[[neighbor_col]]
    ])

In [ ]:
adata_cr_mpr.obs = pd.merge(
    adata_cr_mpr.obs, df_neighbor_CR_CAF_mpr,
    right_index=True, left_index=True, how="left",
)

adata_cr_mpr.obs["CAF_annotation"] = "Other"
adata_cr_mpr.obs.loc[
    adata_cr_mpr.obs["subcluster"].isin(CR_CAF_SUBCLUSTERS), "CAF_annotation"
] = "CR CAF"

adata_cr_mpr.obs["CR_CAF_neighbor_Tumor_CAF"] = adata_cr_mpr.obs["CR CAF_neighbor_Tumor"]
adata_cr_mpr.obs["CR_CAF_neighbor_Tumor_CAF"] = adata_cr_mpr.obs["CR_CAF_neighbor_Tumor_CAF"].astype(str)
adata_cr_mpr.obs.loc[
    adata_cr_mpr.obs["CAF_annotation"] == "CR CAF", "CR_CAF_neighbor_Tumor_CAF"
] = "CR CAF"
adata_cr_mpr = adata_cr_mpr[adata_cr_mpr.obs["CR_CAF_neighbor_Tumor_CAF"].sort_values(ascending=False).index]

### Spatial visualization

In [ ]:
sq.pl.spatial_scatter(
    adata_cr_mpr[
        (adata_cr_mpr.obs["Sample"] == "Pt-14_Resection")
        & (adata_cr_mpr.obs["CR_CAF_neighbor_Tumor_CAF"] != "Other")
    ],
    library_id="spatial", shape=None, size=1,
    color=["CR_CAF_neighbor_Tumor_CAF"],
    wspace=0.4,
    palette=matplotlib.colors.ListedColormap(["red", "black", "blue"]),
)

### DEG analysis + volcano plot

In [ ]:
sc.tl.rank_genes_groups(
    adata_cr_mpr, "CR_CAF_neighbor_Tumor_CAF",
    groups=["Neighbor"], reference="Not_neighbor", method="wilcoxon",
)
degs_cr_mpr = sc.get.rank_genes_groups_df(adata_cr_mpr, group="Neighbor")
degs_cr_mpr = classify_degs(degs_cr_mpr)

plot_volcano(
    degs_cr_mpr, group_name="Neighbor", reference_name="Not_neighbor",
    xlim=(-4, 4), n_top_up=4, n_top_down=20,
    save_path=os.path.join(FIGURES_DIR, "Neighbor_CR_CAF_MPR_neighbor_volcano_integrate.png"),
)

## Other CAF Neighbor Analysis

In [ ]:
adata_other = adata_full[
    (adata_full.obs["Timepoint"] == "Resection") & (adata_full.obs["Response"] == "SD")
].copy()

df_neighbor_other_CAF = pd.DataFrame()
for sample in adata_other.obs["Sample"].unique():
    adata_sample = adata_other[adata_other.obs["Sample"] == sample].copy()
    adata_sample, neighbor_col = identify_caf_neighbors(
        adata_sample, OTHER_CAF_SUBCLUSTERS, "non-NR CAF"
    )

    sq.pl.spatial_scatter(
        adata_sample,
        library_id="spatial", shape=None,
        color=[neighbor_col],
        wspace=0.4,
        palette=matplotlib.colors.ListedColormap(["yellow", "red", "blue"]),
        save=os.path.join(FIGURES_DIR, f"neighbor_non-NR_CAF_{sample}.png"),
    )
    df_neighbor_other_CAF = pd.concat([
        df_neighbor_other_CAF, adata_sample.obs[[neighbor_col]]
    ])

In [ ]:
adata_other.obs = pd.merge(
    adata_other.obs, df_neighbor_other_CAF,
    right_index=True, left_index=True, how="left",
)

adata_other.obs["CAF_annotation"] = "Other"
adata_other.obs.loc[
    adata_other.obs["subcluster"].isin(OTHER_CAF_SUBCLUSTERS), "CAF_annotation"
] = "non-NR CAF"

adata_other.obs["non-NR_CAF_neighbor_Tumor_CAF"] = adata_other.obs["non-NR CAF_neighbor_Tumor"]
adata_other.obs["non-NR_CAF_neighbor_Tumor_CAF"] = adata_other.obs["non-NR_CAF_neighbor_Tumor_CAF"].astype(str)
adata_other.obs.loc[
    adata_other.obs["CAF_annotation"] == "non-NR CAF", "non-NR_CAF_neighbor_Tumor_CAF"
] = "non-NR CAF"
adata_other = adata_other[adata_other.obs["non-NR_CAF_neighbor_Tumor_CAF"].sort_values(ascending=False).index]

In [ ]:
# Merge NR CAF neighbor info for comparison
adata_other.obs = pd.merge(
    adata_other.obs, adata_nr.obs["NR_CAF_neighbor_Tumor_CAF"],
    right_index=True, left_index=True, how="left",
)

adata_other.obs["non-NR_CAF_neighbor_Tumor_CAF"] = adata_other.obs["non-NR_CAF_neighbor_Tumor_CAF"].astype(str)
adata_other.obs.loc[
    adata_other.obs["NR_CAF_neighbor_Tumor_CAF"] == "Neighbor", "non-NR_CAF_neighbor_Tumor_CAF"
] = "NR CAF neighbor"
adata_other.obs.loc[
    adata_other.obs["NR_CAF_neighbor_Tumor_CAF"] == "NR CAF", "non-NR_CAF_neighbor_Tumor_CAF"
] = "NR CAF"
adata_other.obs.loc[
    adata_other.obs["non-NR_CAF_neighbor_Tumor_CAF"] == "Neighbor", "non-NR_CAF_neighbor_Tumor_CAF"
] = "non-NR CAF Neighbor"

adata_other.obs["non-NR_CAF_neighbor_Tumor_CAF"] = pd.Categorical(
    adata_other.obs["non-NR_CAF_neighbor_Tumor_CAF"],
    categories=["Not_neighbor", "non-NR CAF Neighbor", "NR CAF neighbor", "NR CAF", "non-NR CAF", "Other"],
)
adata_other = adata_other[adata_other.obs["non-NR_CAF_neighbor_Tumor_CAF"].sort_values().index]

### Spatial visualization

In [ ]:
sq.pl.spatial_scatter(
    adata_other[
        (adata_other.obs["Sample"] == "Pt-8_Resection")
        & (adata_other.obs["non-NR_CAF_neighbor_Tumor_CAF"] != "Other")
    ],
    library_id="spatial", shape=None, size=1,
    color=["non-NR_CAF_neighbor_Tumor_CAF"],
    wspace=0.4,
    palette=matplotlib.colors.ListedColormap(["gray", "blue", "red", "orange", "lightblue"]),
)

### DEG analysis + volcano plot

In [ ]:
group_name = "NR CAF neighbor"
sc.tl.rank_genes_groups(
    adata_other, "non-NR_CAF_neighbor_Tumor_CAF",
    groups=[group_name], reference="non-NR CAF Neighbor", method="wilcoxon",
)
degs_other = sc.get.rank_genes_groups_df(adata_other, group=group_name)
degs_other = classify_degs(degs_other, up_color="red", down_color="blue")

plot_volcano(
    degs_other, group_name=group_name, reference_name="Non-NR CAF neighbor",
    xlim=(-3, 3), n_top_up=20, n_top_down=20,
    up_color="red", down_color="blue",
    save_path=os.path.join(FIGURES_DIR, f"{group_name}_nonNR_CAF_neighbor_volcano_integrate.png"),
)

In [ ]:
print("Other CAF analysis - downregulated DEGs:")
for gene in degs_other[degs_other["color"] == "blue"]["names"]:
    print(gene)

## CAF Each Patient Analysis

### Load high-resolution filtered data and grid

In [ ]:
adata_high_filtered_caf = ad.read_h5ad(os.path.join(DATA_DIR, "integrate_adata_filtered_caf.h5ad"))
adata_high_filtered_tumor_normal = ad.read_h5ad(os.path.join(DATA_DIR, "integrate_adata_filtered_tumor_normal.h5ad"))

adata_high_filtered_caf.obs["Cell_type"] = adata_high_filtered_caf.obs["leiden"]
adata_high_filtered_tumor_normal.obs["Cell_type"] = adata_high_filtered_tumor_normal.obs["Cell_type_integrate"]

adata_high_filtered = ad.concat([adata_high_filtered_caf, adata_high_filtered_tumor_normal])
del adata_high_filtered_caf, adata_high_filtered_tumor_normal

adata_high_filtered.uns["spatial"] = adata_high_filtered.obsm["spatial"]
print(f"High-filtered combined: {adata_high_filtered.shape}")

### Per-patient gridding and distance calculation

In [ ]:
flag = 0
for patient in PATIENTS:
    adata_hf = adata_high_filtered[
        (adata_high_filtered.obs["Patient"] == patient)
        & (adata_high_filtered.obs["Timepoint"] == "Resection")
    ]

    N_COL = int((adata_hf.obs.imagecol.max() - adata_hf.obs.imagecol.min()) / 10)
    N_ROW = int((adata_hf.obs.imagerow.max() - adata_hf.obs.imagerow.min()) / 10)
    grid = st.tl.cci.grid(
        adata_hf, n_row=N_ROW, n_col=N_COL, n_cpus=32,
        verbose=False, use_label="Cell_type",
    )

    if ("Tumor" in grid.obs["Cell_type"].unique()) and ("Normal" in grid.obs["Cell_type"].unique()):
        grid.obs = pd.merge(
            grid.obs,
            pd.get_dummies(grid.obs["Cell_type"], dtype=int).astype(float).replace(0, -1),
            right_index=True, left_index=True, how="left",
        )

        # Calculate distance from Tumor surface
        grid = calculate_distance(grid, use_label="Tumor")
        df_region = pd.DataFrame(
            np.array(grid.shortest["euclidean"]).reshape(N_ROW, N_COL).T.reshape(N_ROW * N_COL),
            index=["grid_" + str(i + 1) for i in range(N_ROW * N_COL)],
            columns=["euclidean_tumor"],
        )
        grid.obs = pd.merge(grid.obs, df_region, right_index=True, left_index=True, how="left")

        # Calculate distance from Normal surface
        grid = calculate_distance(grid, use_label="Normal")
        df_region = pd.DataFrame(
            np.array(grid.shortest["euclidean"]).reshape(N_ROW, N_COL).T.reshape(N_ROW * N_COL),
            index=["grid_" + str(i + 1) for i in range(N_ROW * N_COL)],
            columns=["euclidean_normal"],
        )
        grid.obs = pd.merge(grid.obs, df_region, right_index=True, left_index=True, how="left")

        # CAF annotation
        grid.obs.loc[
            grid.obs["Cell_type"].isin(["CAF c0", "CAF c3", "CAF c4"]), "CAF annotation"
        ] = "NR CAF"
        grid.obs.loc[grid.obs["Cell_type"].isin(["CAF c1"]), "CAF annotation"] = "CR CAF"
        grid.obs["CAF annotation"] = grid.obs["CAF annotation"].fillna("Other")
        grid.obs["Patient"] = patient

        if flag == 0:
            grid_res = grid
            flag += 1
        else:
            grid_res = ad.concat([grid_res, grid])

print(f"Grid results: {grid_res.shape}")

### Annotate fibroblasts by distance

In [ ]:
grid_res.obs.loc[
    (grid_res.obs["Cell_type"].str.contains("CAF"))
    & (grid_res.obs["euclidean_normal"] <= 3)
    & (grid_res.obs["euclidean_normal"] > 0)
    & (grid_res.obs["euclidean_tumor"] > 3),
    "CAF_annotation_",
] = "Fibroblast"

grid_res.obs.loc[
    grid_res.obs["Cell_type"].isin(["CAF c0", "CAF c3", "CAF c4"]),
    "CAF_annotation_",
] = "NR CAF"

print(grid_res.obs["CAF_annotation_"].value_counts())

### Per-patient DEG and pathway enrichment

In [ ]:
df_caf = pd.DataFrame()
for sample in grid_res.obs["Patient"].unique():
    adata_sample = grid_res[grid_res.obs["Patient"] == sample]

    sc.tl.rank_genes_groups(
        adata_sample, "CAF_annotation_",
        groups=["NR CAF"], reference="Fibroblast", method="wilcoxon",
    )
    degs = sc.get.rank_genes_groups_df(adata_sample, group="NR CAF")
    degs = classify_degs(degs)
    deg_ls = degs[degs["color"] == "red"]["names"].tolist()

    if len(deg_ls) == 0:
        continue

    enr = gp.enrichr(
        gene_list=deg_ls,
        gene_sets=["MSigDB_Hallmark_2020", "KEGG_2021_Human"],
        organism="human",
        outdir=None,
    )
    df_temp = enr.results.copy()
    df_temp["group"] = sample
    df_caf = pd.concat([df_caf, df_temp])

print(f"CAF per-patient pathway results: {df_caf.shape}")

### Pathway enrichment bubble plot (KEGG)

In [ ]:
if not df_caf.empty:
    plot_pathway_bubble(
        df_caf,
        gene_set_filter="KEGG_2021_Human",
        figsize=(5, 8),
        save_path=os.path.join(FIGURES_DIR, "caf_per_patient_KEGG_bubble.png"),
    )